# 5. El modelo y cómo se mide

Un solo LightGBM con todas las variables anteriores. El código está en `modelo.py`.

Este notebook no entrena, solo lee los resultados de `registro/runs.jsonl`. Para volver a medir, correr `python correr.py validar`, que tarda bastante.

In [1]:
import sys
import warnings

sys.path.insert(0, "..")  # Los módulos están en la raíz del proyecto
warnings.filterwarnings("ignore")
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import modelo
import pruebas

corridas = modelo.leer()
final = [r for r in corridas if r.get("nota") == "modelo final: lightgbm con todas las variables"][-1]
pd.Series(modelo.PARAMETROS, name="valor").to_frame()

,valor
n_estimators,1500.000000
learning_rate,0.034310
num_leaves,149.000000
min_child_samples,5.000000
colsample_bytree,0.687459
subsample,0.834477
subsample_freq,1.000000
reg_lambda,0.078270
reg_alpha,0.029235


## Cómo se valida

5 folds repetidos con 2 semillas. Dentro de cada fold, el target encoding se recalcula usando solo las etiquetas de entrenamiento, igual que al predecir el test. Todas las corridas `validacion` usan las mismas particiones, así que se pueden comparar de a pares.

In [2]:
aciertos = np.array(final["accuracy_por_fold"])
print(f"Corrida {final['id']}")
print(f"{final['features']['n']} variables")
print(f"Accuracy de {100 * aciertos.mean():.3f}%  (peor fold {100 * aciertos.min():.3f}%, mejor fold {100 * aciertos.max():.3f}%)")

Corrida 20261104-131600-validacion-lightgbm
183 variables
Accuracy de 94.700%  (peor fold 94.521%, mejor fold 94.929%)


## Cómo cambió el modelo

Validación y leaderboard público de cada submission anotada. Sirve para ver si la validación predice lo que pasa en Kaggle, que es la mejor señal de que no hay sobreajuste.

In [3]:
puntajes = {r["submission"]: r["leaderboard_publico"] for r in corridas if r["tipo"] == "leaderboard"}
subidas = pd.DataFrame([
    {"submission": r["id"], "validación": r["accuracy_validacion"], "leaderboard": puntajes[r["id"]]}
    for r in corridas if r["tipo"] == "submission" and r["id"] in puntajes
])
subidas["diferencia en puntos"] = (100 * (subidas["validación"] - subidas["leaderboard"])).round(3)
subidas.round(5)

,submission,validación,leaderboard,diferencia en puntos
0,20261010-195207-submission-lightgbm,0.94288,0.94352,-0.064
1,20261011-180636-submission-ensamble,0.94374,0.94390,-0.016
2,20261025-212849-submission-ensamble,0.94824,0.94747,0.077
3,20261102-120209-submission-ensamble,0.94739,0.94761,-0.022


## Por qué un solo modelo

Promediar cuatro modelos dio apenas más que LightGBM solo, y la diferencia no se distingue del azar (D50). Se eligió lo más simple.

In [4]:
por_id = {r["id"]: r for r in corridas}
cuatro = por_id["20261102-115951-validacion-ensamble"]["accuracy_por_fold"]
solo = por_id["20261102-134308-validacion-lightgbm"]["accuracy_por_fold"]
dif, error, p = pruebas.t_corregido(cuatro, solo)
print(f"Cuatro modelos menos LightGBM da {100 * dif:+.3f} puntos, error {100 * error:.3f}, p sin ajustar {p:.3f}")

Cuatro modelos menos LightGBM da +0.038 puntos, error 0.024, p sin ajustar 0.149
